- 001 hello world
- 002 add citation overlap (not defined for most!)
- 003 add field overlap
- 004 added govgrant overlap & institution overlap
- 005 include Josh's hand-check
- 006 incorporate related-documents to find patent "families" for m:1 PPPs
- 007 switch from daysdiffnoabs to daysdiff in hope of distinguishing ibetween pre- vs. post- paper pairs in the model 
- 008 after upgrading cosine simliarity and actually applying the m:1 filter. also create ppp_score=5
- 009 disable ppp_score=5 for now but keep the cosine similarity and m:1 filters.
- 009 ALSO parameterized the adjustment for being more than 180 days apart unless there are 3 authors/inventors overlappin
- 010 fold in the Grok-4 LLM rescoring (formerly applyLLMadjustment002, now retired) and promote Grok 'A' pairs to 5 BEFORE the score-0 drop, so an A-rated pair that the forest scores below 0.75 is no longer discarded (962 pairs in the 2026-08 run). ppp_score_rf keeps the pre-LLM score so the patent-family demotion still keys off the random-forest band exactly as in 009. This notebook now writes finalppps.csv, finalpppsadjusted_prescored.csv and finalpppsadjusted.csv.
- 010 ALSO lowered the score-4 (mid/high candidate) cutoff from 0.90 to 0.85 (score4_cutoff in the first cell): the seeded forest puts ~40% fewer pairs above 0.9 than the unseeded Dec-2025 forest did, which had shrunk the mid tier from 41k to 26k; 0.85 restores ~41.5k with hand-check precision flat at ~85%.


In [ ]:
from ppp_paths import databulk, dataraw, dataint, datafin, dataconf
graphoutputdir = '../../../Apps/Overleaf/ppprev1/figures/'
confirmed_confidence_thresh = 3
disconfirmed_confidence_thresh = 3
redo_pat_families = True
max_magids_for_patent = 3
include_daysdiffnoabs = True
include_daysdiff = False
readjust = True   # LLM step: demote a post-shift 4 to 3 when the patent precedes the paper by >180 days and <3 authors overlap (carried over from applyLLMadjustment002)
score4_cutoff = 0.85   # 010: was 0.9 through 009. Lowered so the mid tier is back near its Dec-2025 size (~41k); pred_proba is quantized to 0.01 (100 trees)


In [ ]:
import pandas as pd
import numpy as np
import unicodedata 
import re
from tqdm import tqdm


In [ ]:
possiblepairs = pd.read_parquet(dataint + 'potentialpairswithinwindow.parquet')
possiblepairs = possiblepairs.dropna()
possiblepairs

In [ ]:
# test = pd.read_csv(dataint + 'df_handcoded_henry.csv', dtype=str)
# print(test)
# missingpairs = pd.merge(test, possiblepairs, on=['magid', 'patent_id'], how='inner')
# missingpairs = missingpairs.dropna()
# missingpairs 


In [ ]:
contentsimilarity = pd.read_parquet(dataint + 'possiblepairs_contentsimliarity.parquet')
contentsimilarity = contentsimilarity.rename(columns={'work_id': 'magid'})
contentsimilarity = contentsimilarity.dropna()
contentsimilarity.drop_duplicates(subset=['magid', 'patent_id'], inplace=True)
# contentsimilarity


In [ ]:
institutionoverlap = pd.read_parquet(dataint + 'possiblepairs_institutionoverlap.parquet')
# institutionoverlap

In [ ]:
govgrantoverlap = pd.read_parquet(dataint + 'possiblepairs_govoverlap.parquet')
# govgrantoverlap

In [ ]:
contributoroverlap = pd.read_parquet(dataint + 'possiblepairs_contributoroverlap.parquet')
# contributoroverlap

In [ ]:
fieldoverlap = pd.read_parquet(dataint + 'possiblepairs_fieldoverlap.parquet')
# fieldoverlap                            

In [ ]:
citationoverlap = pd.read_parquet(dataint + 'possiblepairs_citationoverlap.parquet')
# citationoverlap 

In [ ]:
plagiarism = pd.read_parquet(dataint + 'possiblepairs_selfplagiarism.parquet')
# plagiarism

In [ ]:
# missing_rows = possiblepairs[possiblepairs.isna().any(axis=1)]
# missing_rows
# missing_counts = possiblepairs.isna().sum()
# print(missing_counts)

In [ ]:
plagiarism

In [ ]:
possiblepairs = pd.merge(possiblepairs, contentsimilarity, on=['magid', 'patent_id'],  how='left')
print(possiblepairs.shape)
possiblepairs = pd.merge(possiblepairs, contributoroverlap, on=['magid', 'patent_id'], how='left')
print(possiblepairs.shape)
possiblepairs = pd.merge(possiblepairs, fieldoverlap, on=['magid', 'patent_id'], how='left')
print(possiblepairs.shape)
possiblepairs = pd.merge(possiblepairs, citationoverlap, on=['magid', 'patent_id'], how='left')
print(possiblepairs.shape)
possiblepairs = pd.merge(possiblepairs, plagiarism, on=['magid', 'patent_id'], how='left')
print(possiblepairs.shape)
possiblepairs = pd.merge(possiblepairs, govgrantoverlap, on=['magid', 'patent_id'], how='left')
print(possiblepairs.shape)
possiblepairs = pd.merge(possiblepairs, institutionoverlap[['magid', 'patent_id', 'institution_similarity']], on=['magid', 'patent_id'], how='left')
print(possiblepairs.shape)
possiblepairs

In [ ]:
# test = pd.read_csv(dataint + 'df_handcoded_henry.csv', dtype=str)
# print(test.shape)
# missingpairs = pd.merge(test, possiblepairs, on=['magid', 'patent_id'], how='inner')
# missingpairs = missingpairs.dropna()
# missingpairs.shape


In [ ]:
# missing_title = possiblepairs['title_similarity'].isna().sum()
# print(f"Num missing titlesim: {missing_title}")
# missing_abstract = possiblepairs['abstract_similarity'].isna().sum()
# print(f"Num missing abstractsim: {missing_abstract}")
# missing_authoroverlap = possiblepairs['authors in common'].isna().sum()
# print(f"Num missing authoroverlap: {missing_authoroverlap}")
# missing_inventoroverlap = possiblepairs['proportion inventors'].isna().sum()
# print(f"Num missing authoroverlap: {missing_inventoroverlap}")
# missing_plagiarism = possiblepairs['plagiarism_percentage'].isna().sum()
# print(f"Num missing plagiarism: {missing_plagiarism}")
possiblepairs = possiblepairs.dropna()
possiblepairs = possiblepairs.drop_duplicates(subset=['magid', 'patent_id'])
possiblepairs.shape

In [ ]:
# missingpair = possiblepairs[((possiblepairs['magid']=='786684375') & (possiblepairs['patent_id']=='6463793')) | ((possiblepairs['magid']=='128307627') & (possiblepairs['patent_id']=='9788078'))]
# missingpair

positive training set. do it two ways, first with the self-plagiarism, then with the hand-annotated ones. also combine

In [ ]:
df_pos_plagiarism_ORIGINAL = possiblepairs
df_pos_plagiarism_ORIGINAL = df_pos_plagiarism_ORIGINAL[df_pos_plagiarism_ORIGINAL["plagiarism_percentage"] > 10]
df_pos_plagiarism_ORIGINAL = df_pos_plagiarism_ORIGINAL[df_pos_plagiarism_ORIGINAL["plagiarism_group_words_3"] > 9] 
# df_pos_plagiarism_ORIGINAL.to_csv(dataint + "PPP_positive_training_setPLAGIARISM.tsv" , sep = "\t")
df_pos_plagiarism_ORIGINAL

do a tighter cutoff using the 75th percentile. you can push it to 90th but then it is not much for training

In [ ]:
df_pos_plagiarism_TIGHTER = possiblepairs
# #mean
# df_pos_plagiarism_TIGHTER = df_pos_plagiarism_TIGHTER[df_pos_plagiarism_TIGHTER["plagiarism_percentage"] > 27]
# df_pos_plagiarism_TIGHTER = df_pos_plagiarism_TIGHTER[df_pos_plagiarism_TIGHTER["plagiarism_group_words_3"] > 26] 
#75th
df_pos_plagiarism_TIGHTER = df_pos_plagiarism_TIGHTER[df_pos_plagiarism_TIGHTER["plagiarism_percentage"] > 25]
df_pos_plagiarism_TIGHTER = df_pos_plagiarism_TIGHTER[df_pos_plagiarism_TIGHTER["plagiarism_group_words_3"] > 35] 
#90th
# df_pos_plagiarism_TIGHTER = df_pos_plagiarism_TIGHTER[df_pos_plagiarism_TIGHTER["plagiarism_percentage"] > 63]
# df_pos_plagiarism_TIGHTER = df_pos_plagiarism_TIGHTER[df_pos_plagiarism_TIGHTER["plagiarism_group_words_3"] > 62] 

df_pos_plagiarism_TIGHTER.to_csv(dataint + "PPP_positive_training_setPLAGIARISMtighter.tsv" , sep = "\t")
df_pos_plagiarism_TIGHTER

- Cornell PhDs, only validated as very few were invalidated

In [ ]:
df_handcoded_cornellphds = pd.read_csv(dataraw + "cornell_phd_audit_of_pairs.csv", encoding='unicode_escape')
df_handcoded_cornellphds.columns.values[0] = 'magid'
df_handcoded_cornellphds.columns.values[3] = 'patent_id'
# df_handcoded_cornellphds = df_handcoded_cornellphds[['magid', 'patent_id', 'response1', 'response2']]
df_handcoded_cornellphds['validated'] = ((df_handcoded_cornellphds['response1']=='yes') | (df_handcoded_cornellphds['response2']=='yes')) 
#| ((df_handcoded_cornellphds['response1']=='yes') & (df_handcoded_cornellphds['response2']=='maybe')) | ((df_handcoded_cornellphds['response1']=='maybe') & (df_handcoded_cornellphds['response2']=='yes'))
df_handcoded_cornellphds = df_handcoded_cornellphds[df_handcoded_cornellphds['validated']==True]
df_handcoded_cornellphds = df_handcoded_cornellphds[['magid', 'patent_id']]
df_handcoded_cornellphds['patent_id'] = df_handcoded_cornellphds['patent_id'].str.removeprefix('US-')
df_handcoded_cornellphds['magid'] = df_handcoded_cornellphds['magid'].astype(str)
df_handcoded_cornellphds['patent_id'] = df_handcoded_cornellphds['patent_id'].astype(str)
df_handcoded_cornellphds = df_handcoded_cornellphds.drop_duplicates(subset=['magid', 'patent_id'])
df_handcoded_cornellphds.dropna(inplace=True)
print(df_handcoded_cornellphds.shape)
# df_handcoded_cornellphds

In [ ]:
inspect = df_handcoded_cornellphds.merge(possiblepairs, on=['magid', 'patent_id'], how='left')
inspect
inspect.describe(percentiles=[.5, .75, .9, .95, .99])

In [ ]:
df_handcoded_josh = pd.read_csv(dataraw + 'Check Pairs_Josh.csv')
# df_handcoded_henry = pd.read_csv(dataraw + 'mag_patent_ppp_Henry.csv', encoding='unicode_escape')
df_handcoded_josh = df_handcoded_josh.rename(columns={'patent': 'patent_id', 'Patent-paper pair\n(Confidence level 0 to 5)': 'confidence', 'Coverage\n(2- full, 1- partial, 0- not a PPP)':'coverage'})
df_handcoded_josh = df_handcoded_josh[['magid', 'patent_id', 'confidence', 'coverage']]
df_handcoded_josh['patent_id'] = df_handcoded_josh['patent_id'].str.removeprefix('US-')
df_handcoded_josh['magid'] = df_handcoded_josh['magid'].astype(str)
df_handcoded_josh['patent_id'] = df_handcoded_josh['patent_id'].astype(str)
# df_handcoded_henry_confirmed = df_handcoded_henry[df_handcoded_henry['coverage']==2]
# df_handcoded_henry_disconfirmed = df_handcoded_henry[df_handcoded_henry['coverage']==0]
df_handcoded_josh_confirmed = df_handcoded_josh[df_handcoded_josh['confidence']>confirmed_confidence_thresh]
print(df_handcoded_josh_confirmed.shape)
df_handcoded_josh_disconfirmed = df_handcoded_josh[df_handcoded_josh['confidence']<disconfirmed_confidence_thresh]
print(df_handcoded_josh_disconfirmed.shape)
df_handcoded_josh_disconfirmed


In [ ]:
df_handcoded_henry = pd.read_csv(dataraw + 'mag_patent_ppp_Henry.csv', encoding='unicode_escape')
df_handcoded_henry = df_handcoded_henry.rename(columns={'Magid': 'magid', 'Patent': 'patent_id', 'Patent-paper pair\n(Confidence level 0 to 5)': 'confidence', 'Coverage\n(2- full, 1- partial, 0- not a PPP)':'coverage'})
df_handcoded_henry.columns
df_handcoded_henry = df_handcoded_henry[['magid', 'patent_id', 'confidence', 'coverage']]
df_handcoded_henry['patent_id'] = df_handcoded_henry['patent_id'].str.removeprefix('US-')
df_handcoded_henry['magid'] = df_handcoded_henry['magid'].astype(str)
df_handcoded_henry['patent_id'] = df_handcoded_henry['patent_id'].astype(str)
# df_handcoded_henry_confirmed = df_handcoded_henry[df_handcoded_henry['coverage']==2]
# df_handcoded_henry_disconfirmed = df_handcoded_henry[df_handcoded_henry['coverage']==0]
df_handcoded_henry_confirmed = df_handcoded_henry[df_handcoded_henry['confidence']>confirmed_confidence_thresh]
df_handcoded_henry_disconfirmed = df_handcoded_henry[df_handcoded_henry['confidence']<disconfirmed_confidence_thresh]


In [ ]:
import pandas as pd
cross_tab = pd.crosstab(df_handcoded_henry['coverage'], df_handcoded_henry['confidence'])

print(cross_tab)

In [ ]:
#combine Henry and Cornell Phds for validated
df_confirmed = pd.concat([df_handcoded_cornellphds, df_handcoded_henry_confirmed, df_handcoded_josh_confirmed], ignore_index=True)
df_confirmed = df_confirmed[['magid', 'patent_id']]
df_confirmed = df_confirmed.drop_duplicates(subset=['magid', 'patent_id'])
print(f"confirmed before dropping missing values: {df_confirmed.shape}")
df_confirmed = df_confirmed.dropna()
print(f"confirmed after dropping missing values: {df_confirmed.shape}")
df_confirmed = df_confirmed.merge(possiblepairs, on=['magid', 'patent_id'], how='left')
df_confirmed['byhand'] = 1
print(df_confirmed.shape)


In [ ]:
#combine Henry and josh disconfirmed (only a couple cornellphds)
df_disconfirmed = pd.concat([df_handcoded_henry_disconfirmed, df_handcoded_josh_disconfirmed], ignore_index=True)
df_disconfirmed = df_disconfirmed.merge(possiblepairs, on=['magid', 'patent_id'], how='left')
df_disconfirmed['byhand'] = 1


In [ ]:
# test = pd.read_csv(dataint + 'df_handcoded_henry.csv', dtype=str)
# print(test)
print(df_disconfirmed.shape)
missingpairs = pd.merge(df_disconfirmed, possiblepairs, on=['magid', 'patent_id'], how='inner')
missingpairs = missingpairs.dropna()
print(missingpairs .shape)


In [ ]:
df_disconfirmed = df_disconfirmed.drop_duplicates(subset=['magid', 'patent_id'])
print(f"disconfirmed before dropping missing values: {df_disconfirmed.shape}")
df_disconfirmed = df_disconfirmed.dropna()
print(f"disconfirmed after dropping missing values: {df_disconfirmed.shape}")
# df_confirmed.describe()

In [ ]:
print(df_confirmed.shape)
print(df_disconfirmed.shape)
# df_confirmed

In [ ]:
df_pos_plagiarism_TIGHTER['byhand']= 0

In [ ]:
df_pos_plagiarism_TIGHTERhand = pd.concat([df_pos_plagiarism_TIGHTER, df_confirmed], ignore_index=True)

In [ ]:
df_pos_plagiarism_TIGHTERhand.columns
# df_confirmed.columns

Negative training set: pairs more than 10 years apart. these will only be from self-cites because we clipped the pqr pairs at 2 years

In [ ]:
possiblepairs

In [ ]:
## save "non PPPs" (more than 10 years appart)
# table_neg = pppapppapers2analyze[pppapppapers2analyze['difference_patent_app_paper'] > pd.Timedelta(days=365*10)].sample(frac=0.1)
table_neg = possiblepairs[possiblepairs['daysdiff'] >365*10].sample(frac=0.1, random_state=42)
# table_neg.to_csv(dataint + "PPP_negative_training_set.tsv" , sep = "\t")
table_neg_original = table_neg.copy()
table_neg_tighter = table_neg.sample(n=len(df_pos_plagiarism_TIGHTER), random_state=42)
# table_neg.describe()
table_neg_tighterhand = pd.concat([table_neg_tighter, df_disconfirmed], ignore_index=True)
table_neg_tighterhand = table_neg_tighterhand.drop_duplicates(subset=['magid', 'patent_id'])
print(table_neg_tighterhand.shape)
table_neg_hand = df_disconfirmed.copy()


final training samples

In [ ]:
table_neg_original 
table_neg_tighter 
table_neg_tighterhand 
table_neg_hand 

df_pos_plagiarism_ORIGINAL
df_pos_plagiarism_TIGHTER
df_pos_plagiarism_TIGHTERhand  
df_confirmed 
df_confirmed.shape
df_disconfirmed.shape

OK this is hacky. set df_pos and table_neg to whichever ones you are going to use

In [ ]:
print(df_pos_plagiarism_TIGHTER.shape)
print(df_pos_plagiarism_TIGHTERhand.shape)
print(table_neg_tighter.shape)
print(table_neg_tighterhand.shape)


In [ ]:
table_neg_tighterhand
df_pos_plagiarism_TIGHTERhand

In [ ]:
##original paper
df_pos = df_pos_plagiarism_ORIGINAL
df_neg = table_neg_original
## hand-checked 
df_pos = df_confirmed
df_neg = table_neg_hand
## plagiarism with 75th percentile cutoff s
df_pos = df_pos_plagiarism_TIGHTER
df_neg = table_neg_tighter
## combined 
df_pos = df_pos_plagiarism_TIGHTERhand
df_neg = table_neg_tighterhand


df_pos["output"] = 1
# print(len(df_pos))
# print(len(df_neg))
df_neg["output"] = 0
# df = pd.concat([df_pos , df_neg[100:len(df_pos) + 100]])
df = pd.concat([df_pos , df_neg])

df['byhand'] = df['byhand'].fillna(0)
df


In [ ]:
print(df.shape)
df.value_counts('byhand')

Train the model

In [ ]:
## get accuracy metrics to evaluate Machine Learning models

from sklearn.metrics import confusion_matrix


# Below is the code for calcultating the accuracy, the TPR and the FPR.
def accuracy(y_test,y_pred):
    cm = confusion_matrix(y_test, y_pred)
    accuracy = (cm.ravel()[0]+cm.ravel()[3])/sum(cm.ravel())
    return accuracy

def TPR(y_test,y_pred):
    cm = confusion_matrix(y_test, y_pred)
    TPR = cm[1][1]/(cm[1][1]+cm[1][0])
    return TPR

def FPR(y_test,y_pred):
    cm = confusion_matrix(y_test, y_pred)
    FPR = cm[0][1]/(cm[0][1]+cm[0][0])
    return FPR


def precision(y_test,y_pred):
    cm = confusion_matrix(y_test, y_pred)
    TPR = cm[1][1]/(cm[1][1]+cm[0][1])
    return TPR

def recall(y_test,y_pred):
    cm = confusion_matrix(y_test, y_pred)
    FPR = cm[1][1]/(cm[1][1]+cm[1][0])
    return FPR

In [ ]:
# # Assuming your dataframe is named df and it has a column named 'byhand'
# rows_with_byhand_1 = df[df['byhand'] == 1]
# rows_with_byhand_1 = rows_with_byhand_1.dropna()
# # Display the rows
# print(rows_with_byhand_1)
# rows_with_byhand_1.value_counts('output')

In [ ]:
##correlation matrix

import matplotlib.pyplot as plt
import seaborn as sns

if include_daysdiffnoabs is True:
  df = df[['title_similarity', 'abstract_similarity', 'authors in common',
        'proportion inventors', 'citeoverlapct',   'daysdiffnoabs', 'institution_similarity', 
          'output']]
elif include_daysdiff is True:
    df = df[['title_similarity', 'abstract_similarity', 'authors in common',
            'proportion inventors', 'citeoverlapct', 'daysdiff', 'institution_similarity', 
            'output']]
else:
  df = df[['title_similarity', 'abstract_similarity', 'authors in common',
        'proportion inventors', 'citeoverlapct',    'institution_similarity', 
          'output']]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df.apply(pd.to_numeric , errors='coerce')
df = df.apply(pd.to_numeric , errors='coerce')

plt.figure(figsize=(16, 6))
heatmap = sns.heatmap(df.corr(), vmin=-1, vmax=1, annot=True , center = 0 , cmap="PiYG")
heatmap.set_title('Correlation Heatmap', fontdict={'fontsize':12}, pad=12)
plt.savefig(graphoutputdir + '/correlations.png')

In [ ]:
## training set: 80%, testing set: 20%

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
X = df.drop(["output" ] , axis = 1)
y = df["output"]

#80% of the data from the training set while 20% form the testing set
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
#run the logistic model

import statsmodels.formula.api as smf

training_set = X_train
training_set["output"] = y_train
training_set= training_set.rename(columns = { "proportion inventors" : "proportion_inventors" , "authors in common" : "authors_in_common" } )

#regression_model = smf.logit('output ~ title_similarity + abstract_similarity + authors_in_common + proportion_inventors + distance_inst_assignee  + front + body ' , data=training_set , missing='drop').fit()
# regression_model = smf.logit('output ~ title_similarity + abstract_similarity + authors_in_common + proportion_inventors + distance_inst_assignee  ' , data=training_set , missing='drop').fit()
if include_daysdiffnoabs is True:
    regression_model = smf.logit('output ~ title_similarity + daysdiffnoabs + abstract_similarity + authors_in_common + proportion_inventors + institution_similarity +  citeoverlapct ' , data=training_set , missing='drop').fit()
elif include_daysdiff is True:
    regression_model = smf.logit('output ~ title_similarity + daysdiff + abstract_similarity + authors_in_common + proportion_inventors + institution_similarity +  citeoverlapct ' , data=training_set , missing='drop').fit()

else:
    regression_model = smf.logit('output ~ title_similarity +  abstract_similarity + authors_in_common + proportion_inventors + institution_similarity +  citeoverlapct ' , data=training_set , missing='drop').fit()

# regression_model = smf.logit('output ~ title_similarity + abstract_similarity + authors_in_common + proportion_inventors + institution_similarity + sameinstitutionflag + citeoverlapct + fieldoverlap + samegrant + sameagency + daysdiffnoabs' , data=training_set , missing='drop').fit()

print(regression_model.summary())

In [ ]:
#evaluate the logistic model on the testing set
test_set = X_test
test_set["output"] = y_test
test_set= test_set.rename(columns = { "proportion inventors" : "proportion_inventors" , "authors in common" : "authors_in_common" } )

y_pred = regression_model.predict(test_set).to_numpy()
y_test_log = test_set["output"]
## Baseline model:
y_baseline = 1- np.zeros((len(y_pred)))

print(" The accuracy of the baseline model on the test set is equal to" , accuracy(y_test_log,y_baseline))  
print(" The TPR of the logistic baseline model on the test set is equal to" , TPR(y_test_log,y_baseline))  
print(" The FPR of the logistic baseline model on the test set is equal to" , FPR(y_test_log,y_baseline))  


print(" ")
n = len(y_pred)
y_pred_log = [0]*n
for i in tqdm(range(n)) :
    if y_pred[i] >= 0.50 :
            y_pred_log[i] = 1
print(" The accuracy of the logistic regression method on the test set is equal to" , accuracy(y_test_log,y_pred_log))  
print(" The TPR of the logistic regression method on the test set is equal to" , TPR(y_test_log,y_pred_log))  
print(" The FPR of the logistic regression method on the test set is equal to" , FPR(y_test_log,y_pred_log))  

Decision tree

In [ ]:
X_train = X_train.drop(["output"] , axis = 1)
X_test = X_test.drop(["output"] , axis = 1)

In [ ]:
#run the decision tree model

from sklearn.tree import DecisionTreeClassifier


dtc = DecisionTreeClassifier(min_samples_leaf=8, 
                             ccp_alpha= 0.0015,
                             criterion = 'gini',
                             random_state = 88)

dtc = dtc.fit(X_train, y_train)

#below is the code for plotting the decision tree classifier and counting the number of nodes. 


import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

print('Node count =', dtc.tree_.node_count)
plt.figure(figsize=(18,15))
plot_tree(dtc, 
          feature_names=X_train.columns, 
          class_names=['0','1'], 
          filled=True,
          impurity=True,
          rounded=True,
          fontsize=8
         ) 
plt.show()

In [ ]:
#evaluate the decision tree model on the testing set


y_pred = dtc.predict(X_test)
y_baseline = 1 - np.zeros((len(y_pred)))

print(" The accuracy of the baseline model on the test set is equal to" , accuracy(y_test,y_baseline))  
print(" The TPR of the logistic baseline model on the test set is equal to" , TPR(y_test,y_baseline))  
print(" The FPR of the logistic baseline model on the test set is equal to" , FPR(y_test,y_baseline))  


print( " ")
print(" The accuracy of the decision tree regression method on the test set is equal to" , accuracy(y_test,y_pred))  
print(" The TPR of the decision tree method on the test set is equal to" , TPR(y_test,y_pred))  
print(" The FPR of the decision tree method on the test set is equal to" , FPR(y_test,y_pred))  


random forest

In [ ]:
#run the random forest

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

import pickle

filename = dataint + '/random_forest_PPPs.pkl'
pickle.dump(rf, open(filename, 'wb'))

In [ ]:
y_pred = rf.predict_proba(X_test)


## Baseline model:
y_baseline = np.zeros((len(y_pred)))

print(" The accuracy of the baseline model on the test set is equal to" , accuracy(y_test,y_baseline))  
print(" The TPR of the logistic baseline model on the test set is equal to" , TPR(y_test,y_baseline))  
print(" The FPR of the logistic baseline model on the test set is equal to" , FPR(y_test,y_baseline))  

print(" ") 

n = len(y_pred)
y_pred_rf = [0]*n
for i in range(n) :
    if y_pred[i][1] >= 0.75 :
            y_pred_rf[i] = 1

print(" The accuracy of the random forest method on the test set is equal to" , accuracy(y_test,y_pred_rf))  
print(" The TPR of the random forest method on the test set is equal to" , TPR(y_test,y_pred_rf))  
print(" The FPR of the random forest method on the test set is equal to" , FPR(y_test,y_pred_rf))  

In [ ]:
thresholds = [ 0.75 , 0.8 , 0.85 , 0.9] 
dic_results = {}


for threshold in thresholds:
    
    dic_results[threshold] = {}
    
    
    y_pred = rf.predict_proba(X_test)

    n = len(y_pred)
    y_pred_rf = [0]*n
    for i in range(n) :
        if y_pred[i][1] >= threshold :
                y_pred_rf[i] = 1
                
    dic_results[threshold]["precision"] = precision(y_test,y_pred_rf)*100
    
    dic_results[threshold]["f1"] = 100*2*( precision(y_test,y_pred_rf) * recall(y_test,y_pred_rf) ) / ( precision(y_test,y_pred_rf) + recall(y_test,y_pred_rf))
    
    dic_results[threshold]["recall"] = recall(y_test,y_pred_rf)*100

    print(" The f1 of the random forest method on the test set is equal to" , 2*( precision(y_test,y_pred_rf) * recall(y_test,y_pred_rf) ) / ( precision(y_test,y_pred_rf) + recall(y_test,y_pred_rf) )) 
    print(" The precision of the random forest method on the test set is equal to" , precision(y_test,y_pred_rf))  
    print(" The recall of the random forest method on the test set is equal to" , recall(y_test,y_pred_rf)) 
   
    print(" ")
    print(" ")

this is hardcoded/broken and needs to be tethought

In [ ]:
"""
    [finalppps['pred_proba'] < 0.75, 
     (finalppps['pred_proba'] >= 0.75) & (finalppps['pred_proba'] < 0.8), 
     (finalppps['pred_proba'] >= 0.8) & (finalppps['pred_proba'] < 0.85), 
     (finalppps['pred_proba'] >= 0.85) & (finalppps['pred_proba'] < 0.9), 
     finalppps['pred_proba'] >= 0.9],
"""
pd.DataFrame(dic_results).rename( columns = {  0.75 : "PPP if prediction > 0.75" ,  .8 : "PPP if prediction > 0.8" ,  0.85 : "PPP if prediction > 0.85" ,  0.9: "PPP if prediction > 0.9" } )


In [ ]:
# Get numerical feature importances


#importances = sorted(list(rf.feature_importances_) , reverse = True)
importances = list(rf.feature_importances_)


feature_list = list(X_test.columns)
# List of tuples with variable and importance
feature_importances = [(feature, round(importance, 4)) for feature, importance in zip(feature_list, importances)]
# Sort the feature importances by most important first
feature_importances = sorted(feature_importances, key = lambda x: x[1], reverse = True)
#feature_importances = feature_importances
# Print out the feature and importances 
[print('Variable: {:20} Importance: {}'.format(*pair)) for pair in feature_importances]


In [ ]:
#plot the feature importance 

x_values = list(range(len(importances)))
# Make a bar chart
plt.bar(x_values, sorted(importances , reverse = True), orientation = 'vertical', color = 'r', edgecolor = 'k', linewidth = 1.2 )
# Tick labels for x axis
plt.xticks(x_values, list(np.array(feature_importances)[:,0]), rotation='vertical')
plt.ylim((0,1))
# Axis labels and title
plt.ylabel('Importance'); plt.xlabel('Variable'); plt.title('Variable Importance')
plt.savefig(graphoutputdir + '/variableimportance.png')

In [ ]:
#plot the feature importance 


# List of features sorted from most to least important
sorted_importances = [importance[1] for importance in feature_importances]
sorted_features = [importance[0] for importance in feature_importances]
# Cumulative importances
cumulative_importances = np.cumsum(sorted_importances)
# Make a line graph
plt.plot(x_values, cumulative_importances, 'g-')
plt.ylim((0,1.1))
# Draw line at 95% of importance retained
plt.hlines(y = 0.95, xmin=0, xmax=len(sorted_importances), color = 'r', linestyles = 'dashed')
# Format x ticks and labels
plt.xticks(x_values, sorted_features, rotation = 'vertical')
# Axis labels and title
plt.xlabel('Variable'); plt.ylabel('Cumulative Importance'); plt.title('Cumulative Importance')
plt.savefig(graphoutputdir + '/cumulativeimportance.png')

In [ ]:
#plot the distribution of the results

sns.displot(y_pred[:,1] , binwidth=0.01 )
plt.title("Distribution of the predictions")
plt.xlabel("Prediction")
plt.savefig(graphoutputdir + '/ushapedpredictions.png')

In [ ]:
#plot the ROC curve (TPR as a function of the FPR)

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test, y_pred[:,1])
roc_auc = auc(fpr, tpr)
print(roc_auc)

plt.figure(figsize=(8, 6))
plt.title('ROC Curve', fontsize=18)
plt.xlabel('FPR', fontsize=16)
plt.ylabel('TPR', fontsize=16)
plt.xlim([-0.01, 1.00])
plt.ylim([-0.01, 1.01])
plt.plot(fpr, tpr, lw=3, label='Logistic Regression (area = {:0.2f})'.format(roc_auc))
plt.plot([0, 1], [0, 1], color='navy', lw=3, linestyle='--')
plt.legend(loc='lower right', fontsize=14)
plt.show()
plt.savefig(graphoutputdir + '/roc.png')

In [ ]:
df_export = df.copy()
df_export['test_pred_proba'] = np.nan
df_export['train_pred_proba'] = np.nan
y_pred_proba = rf.predict_proba(X_test)[:, 1]
df_export.loc[X_test.index, 'test_pred_proba'] = y_pred_proba
train_pred_proba = rf.predict_proba(X_train)[:, 1]
df_export.loc[X_train.index, 'train_pred_proba'] = train_pred_proba
df_export['pred_proba'] = df_export['test_pred_proba'].combine_first(df_export['train_pred_proba'])
df_export['pred_proba'] = df_export['test_pred_proba'].combine_first(df_export['train_pred_proba'])
# df_export = df_export[['magid', 'patent_id', 'pred_proba']]


In [ ]:
possiblepairs

need to drop the poential PPPs more than 2 years apart. we kept the self-cites with longer lengths for negative training examples, but now we shed them

In [ ]:
possiblepairs.columns
possiblepairs.to_parquet(dataint + 'possiblepairs.parquet', index=False, compression='snappy')

START HERE IF YOU JUST WANT TO RESCORE

In [ ]:
finalppps = pd.read_parquet(dataint + 'possiblepairs.parquet')
# finalppps = finalppps.drop(columns=['doi'])
# finalppps = finalppps.drop(columns=['plagiarism_group_words_1'])
# finalppps = finalppps.drop(columns=['plagiarism_group_words_3'])
# finalppps = finalppps.drop(columns=['plagiarism_group_words_5'])
# finalppps = finalppps.drop(columns=['plagiarism_percentage'])
finalppps = finalppps[finalppps['daysdiff'] <365*2]
# finalppps = finalppps.drop(columns=['daysdiff'])
# finalppps = finalppps.drop(columns=['selfcite', 'doi', 'plagiarism_group_words_1', 'plagiarism_group_words_3', 'plagiarism_group_words_5', 'plagiarism_percentage', 'sameagency'])
# finalppps = finalppps.rename(columns={'daysdiffnoabs':'daysdiff'})
# finalppps = finalppps[['magid', 'patent_id', 'title_similarity', 'abstract_similarity', 'authors in common', 'proportion inventors', 'daysdiffnoabs', 'citeoverlapct', 'fieldoverlap', 'samegrant', 'institution_similarity']]
print(finalppps.columns)
finalppps = finalppps.dropna()
try:
    X_full = finalppps[X_train.columns]
except KeyError as e:
    print(f"Missing columns: {set(X_train.columns) - set(finalppps.columns)}")
    print(f"Extra columns: {set(finalppps.columns) - set(X_train.columns)}")
    raise
finalppps['pred_proba'] = rf.predict_proba(X_full)[:, 1]

In [ ]:
# // ...existing code...
high_conf = finalppps[finalppps['pred_proba'] > 0.88]['pred_proba']

plt.figure(figsize=(8, 5))
plt.hist(high_conf, bins=30, edgecolor='black')
plt.xlabel('pred_proba')
plt.ylabel('count')
plt.title('Prediction score distribution (pred_proba > 0.8)')
plt.show()
# // ...existing code...

In [ ]:
finalppps['ppp_score'] = np.select(
    [finalppps['pred_proba'] < 0.75, 
     (finalppps['pred_proba'] >= 0.75) & (finalppps['pred_proba'] < 0.8), 
     (finalppps['pred_proba'] >= 0.8) & (finalppps['pred_proba'] < score4_cutoff), 
     (finalppps['pred_proba'] >= score4_cutoff) & (finalppps['pred_proba'] < score4_cutoff),   # empty band, kept so scores stay 0-4
     finalppps['pred_proba'] >= score4_cutoff],
    [0, 1, 2, 3, 4],
    default=0  # Fallback for NaN or unexpected values
)
# drop confidence if patent is more than a year after paper published
finalppps['ppp_score'] = np.where((finalppps['ppp_score']==4) & (finalppps['daysdiffnoabs']>365), 3, finalppps['ppp_score'])
print(finalppps.shape)
finalppps.value_counts(['ppp_score']).sort_index(ascending=False)


In [ ]:
finalppps['plagiarism_percentage'].describe()

In [ ]:
finalppps

In [ ]:
# low_plagiarism

In [ ]:
# Filter and get value counts with percentages
hi_plagiarism = finalppps[finalppps['plagiarism_percentage'] >= 25]
print(f"Total rows with plagiarism_percentage >= 25: {len(hi_plagiarism)}")
print("\nBreakdown by ppp_score:")
print(hi_plagiarism['ppp_score'].value_counts().sort_index())

In [ ]:
# Filter and get value counts with percentages
low_plagiarism = finalppps[finalppps['plagiarism_percentage'] < 25]
print(f"Total rows with plagiarism_percentage < 25: {len(low_plagiarism)}")
print("\nBreakdown by ppp_score:")
print(low_plagiarism['ppp_score'].value_counts().sort_index())

In [ ]:
print(finalppps['ppp_score'].value_counts().sort_index())

In [ ]:
# finalppps = finalppps.drop(columns=['selfcite', 'doi', 'plagiarism_group_words_1', 'plagiarism_group_words_3', 'plagiarism_group_words_5', 'plagiarism_percentage', 'sameagency'])
finalppps = finalppps.drop(columns=['selfcite', 'doi',  'plagiarism_group_words_1',  'plagiarism_group_words_5', 'sameagency'])


In [ ]:
dtypes = {
    'paperid': 'str'              # magid
}
# paperyears = pd.read_csv(databulk + 'paperdates.tsv.zip', sep='\t', header=None, compression='zip')  
paperdates = pd.read_csv(databulk + 'paperdates.zip', sep='\t', dtype=dtypes, parse_dates=[0], date_format='%Y%m%d', header=None, compression='zip')  
# paperdates = pd.read_csv(databulk + 'paperdates.tsv', sep='\t', dtype=dtypes, parse_dates=[0], date_format='%Y%m%d', header=None)  
# paperdates.rename(columns={1: 'paperpubdate', 0: 'magid'}, inplace=True) 
paperdates = paperdates.rename(columns={0: 'magid'})
paperdates = paperdates.rename(columns={1: 'paperpubdate'})
paperdates['magid'] = paperdates['magid'].astype(str)
paperdates.dropna(inplace=True)
paperdates.drop_duplicates(inplace=True)
paperdates                        

In [ ]:
print(finalppps.shape)

In [ ]:
finalppps

drop the zeroes and export

In [ ]:
finalppps['ppp_score'] = finalppps['ppp_score'].astype(int)
print(finalppps.shape)


In [ ]:
#if high self plagiarism, make it confidence score 4
finalppps.loc[(finalppps['plagiarism_percentage'] >= 25), 'ppp_score'] = 4


In [ ]:
#if on the same grant, raise by 1 unless already 4
finalppps.loc[(finalppps['samegrant']==1) & (finalppps['ppp_score'] < 4), 'ppp_score'] += 1

In [ ]:
# finalppps.loc[(finalppps['authors in common']==1) & (finalppps['ppp_score'] > 3), 'ppp_score'] = finalppps['ppp_score'] - 1

In [ ]:
# affected_rows = finalppps[(finalppps['ppp_score'] == 4) & (finalppps['daysdiff'] < -180)].shape[0]
# print(f"Number of rows affected: {affected_rows}")
# finalppps.loc[(finalppps['ppp_score'] == 4) & (finalppps['daysdiff'] < -180), 'ppp_score'] = 3

In [ ]:
bl = 0
if bl == 1:
    affected_rows = finalppps[(finalppps['ppp_score'] == 4) & (finalppps['daysdiffnoabs'] < -180) & (finalppps['authors in common'] < 3)].shape[0]
    print(f"Number of rows affected: {affected_rows}")
    finalppps.loc[(finalppps['ppp_score'] == 4) & (finalppps['daysdiffnoabs'] < -180) & (finalppps['authors in common'] < 3), 'ppp_score'] = 3
    # was 3775 affected with authorsincoomon<3


### 010: merge the Grok-4 assessments and promote 'A' before dropping the zeroes

The assessments (A = confirmed PPP, B = plausible, C = unlikely, D = not a PPP; prompt in `data/raw/ppp_prompt.txt`) were produced once, in Nov 2025, on the score-3/4 bands of the forest current at that time. Because the forest is re-trained on every run, some A-rated pairs now land below the 0.75 floor; 009 discarded them at the score-0 drop before the assessments were ever consulted. Promoting A to 5 here fixes that. `ppp_score_rf` preserves the pre-LLM score for the patent-family step below.

Merged exactly as applyLLMadjustment002 did (no de-duplication of the assessment files; the keep-max de-duplication at the end resolves conflicting labels).

In [ ]:
llmadjustments = pd.concat([
    pd.read_csv(dataraw + 'ppp_grok4_refined_4_allDONOTDELETE.csv', dtype={'magid': str, 'patent_id': str}),
    pd.read_csv(dataraw + 'ppp_grok4_refined_4_allDONOTDELETE_MISSING.csv', dtype={'magid': str, 'patent_id': str}),
    pd.read_csv(dataraw + 'ppp_grok4_refined_3DONOTDELETE.csv', dtype={'magid': str, 'patent_id': str}),
], ignore_index=True)
print(llmadjustments.shape)
print(llmadjustments.value_counts('assessment').sort_values(ascending=False))

finalppps = finalppps.merge(llmadjustments[['magid', 'patent_id', 'assessment']], on=['magid', 'patent_id'], how='left')
finalppps['ppp_score_rf'] = finalppps['ppp_score']   # pre-LLM score; drives the patent-family demotion below

rescued = finalppps[(finalppps['assessment'] == 'A') & (finalppps['ppp_score'] == 0)][['magid', 'patent_id']].drop_duplicates()
print(f"Grok-A pairs below the 0.75 floor that 009 would have discarded: {len(rescued)}")
finalppps.loc[finalppps['assessment'] == 'A', 'ppp_score'] = 5
print(finalppps.value_counts('ppp_score').sort_index())


In [ ]:
finalppps = finalppps[finalppps['ppp_score'] > 0]
print(finalppps.value_counts('ppp_score').sort_index())
#48299

In [ ]:
import pandas as pd
import os
import logging
import time

# Setup logging
logging.basicConfig(filename='family_id_assignment.log', level=logging.INFO, format='%(asctime)s - %(message)s')

# Configuration
# input_file = 'C:\\data\\raw\\g_us_rel_doc.tsv'
# output_file = 'C:\\data\\raw\\patent_family_ids.parquet'

# Union-Find data structure for connected components
class UnionFind:
    def __init__(self):
        self.parent = {}
        self.rank = {}
    
    def make_set(self, x):
        if x not in self.parent:
            self.parent[x] = x
            self.rank[x] = 0
    
    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # Path compression
        return self.parent[x]
    
    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        if px == py:
            return
        # Union by rank
        if self.rank[px] < self.rank[py]:
            self.parent[px] = py
        elif self.rank[px] > self.rank[py]:
            self.parent[py] = px
        else:
            self.parent[py] = px
            self.rank[px] += 1

# Main processing
def assign_family_ids(input_file, output_file):
    start_time = time.time()
    
    try:
        # Load data
        logging.info("Loading g_us_rel_doc.tsv...")
        df = pd.read_csv(input_file, sep='\t', usecols=['patent_id', 'related_doc_number'], dtype=str, compression='zip')
        logging.info(f"Loaded {len(df):,} rows")
        print(f"Loaded {len(df):,} rows")
        
        # Initialize union-find
        uf = UnionFind()
        
        # Get all unique patent IDs
        logging.info("Extracting unique patent IDs...")
        patent_ids = set(df['patent_id']).union(set(df['related_doc_number']))
        for patent_id in patent_ids:
            uf.make_set(patent_id)
        logging.info(f"Found {len(patent_ids):,} unique patent IDs")
        print(f"Found {len(patent_ids):,} unique patent IDs")
        
        # Build families by unioning related patents
        logging.info("Assigning patent families...")
        for _, row in df.iterrows():
            uf.union(row['patent_id'], row['related_doc_number'])
        
        # Create family ID mapping
        family_map = {}
        family_id_counter = 0
        for patent_id in patent_ids:
            root = uf.find(patent_id)
            if root not in family_map:
                family_map[root] = str(family_id_counter)
                family_id_counter += 1
            family_map[patent_id] = family_map[root]
        
        # Create output DataFrame
        output_df = pd.DataFrame({
            'patent_id': list(family_map.keys()),
            'patfamilyid': list(family_map.values())
        })
        output_df['patent_id'] = output_df['patent_id'].astype(str)
        output_df['patent_id'] = output_df['patent_id'].apply(lambda x: x.lstrip('0'))
        output_df = output_df.sort_values('patent_id')  # Sort for consistency
        logging.info(f"Assigned {family_id_counter:,} unique family IDs")
        print(f"Assigned {family_id_counter:,} unique family IDs")
        
        # Save to parquet
        output_df.to_parquet(output_file, engine='pyarrow', index=False)
        logging.info(f"Saved {len(output_df):,} rows to {output_file}")
        print(f"Saved {len(output_df):,} rows to {output_file}")
        
        # Log total time
        total_time = time.time() - start_time
        logging.info(f"Total processing time: {total_time:.2f} seconds")
        print(f"Total processing time: {total_time:.2f} seconds")
        
        return output_df
    
    except Exception as e:
        logging.error(f"Error processing data: {str(e)}")
        print(f"Error: {str(e)}")
        raise

In [ ]:
# redo_pat_families = False
# redo_pat_families = True
if redo_pat_families is True:
    patent_families = assign_family_ids(databulk + 'g_us_rel_doc.tsv.zip', dataint + 'patfamilies.parquet')
    # patent_families = patent_families.rename(columns={'family_id':'patfamilyid'})
    print(patent_families)

In [ ]:
patent_families = pd.read_parquet(dataint + 'patfamilies.parquet')
patent_families['patent_id'] = patent_families['patent_id'].apply(lambda x: x.lstrip('0'))
patent_families['patent_id'] = patent_families['patent_id'].astype(str)
# print(patent_families.value_counts('patfamilyid'))
# patent_families

for high-confidence PPPs where one magid maps to multiple patents but they are not all in the same family, lower confidence by 1

In [ ]:
finalpppspatfamily = pd.merge(finalppps, patent_families, on='patent_id', how='left')
# print(finalpppspatfamilyhiconf)
# 010: key the family check off the pre-LLM score so Grok-A pairs (now 5) still count toward a paper's patents, exactly as in 009
finalpppspatfamilyhiconf = finalpppspatfamily[finalpppspatfamily['ppp_score_rf']==4]
# finalpppspatfamilyhiconf = finalpppspatfamilyhiconf[finalpppspatfamilyhiconf['patfamilyid'].isna()]
# print(finalpppspatfamilyhiconf.shape)
# patentid_counts = finalpppspatfamilyhiconf.groupby('patent_id')['magid'].nunique().reset_index()
# patentid_counts.columns = ['patent_id', 'magid_count']
# # patentid_counts = patentid_counts[patentid_counts['magid_count']>1]
# patentid_counts = patentid_counts.sort_values(by='magid_count', ascending=False)
# patentid_counts.value_counts('magid_count')
finalpppspatfamilyhiconf



In [ ]:
magid_counts = finalpppspatfamilyhiconf.groupby('magid').agg({
    'patent_id': 'nunique',
    'patfamilyid': 'nunique'
}).rename(columns={
    'patent_id': 'patent_count',
    'patfamilyid': 'family_count'
}).reset_index()

# Filter for magids with multiple patent_ids AND multiple family_ids
problem_magids = magid_counts[(magid_counts['patent_count'] > 1) &  (magid_counts['family_count'] != 1)]
problem_magids.loc[:, 'problem_magid'] = True
problem_magids

In [ ]:
patentid_counts = finalpppspatfamilyhiconf.groupby('patent_id')['magid'].nunique().reset_index()
patentid_counts.columns = ['patent_id', 'magid_count']
# patentid_counts = patentid_counts[patentid_counts['magid_count']>1]
patentid_counts = patentid_counts.sort_values(by='magid_count', ascending=False)
patentid_counts.value_counts('magid_count')


problem_patents = patentid_counts[(patentid_counts['magid_count'] > max_magids_for_patent)]
problem_patents.loc[:, 'problem_patent'] = True
problem_patents



In [ ]:
print(finalpppspatfamily.shape)
print(finalpppspatfamily.value_counts('ppp_score'))
finalpppspatfamily = pd.merge(finalpppspatfamily, problem_magids, on='magid', how='left')
finalpppspatfamily = pd.merge(finalpppspatfamily, problem_patents, on='patent_id', how='left')
finalpppspatfamily.loc[(finalpppspatfamily['ppp_score']==4) & (finalpppspatfamily['problem_magid']), 'ppp_score'] = 3
print(finalpppspatfamily.value_counts('ppp_score'))
finalpppspatfamily.loc[(finalpppspatfamily['ppp_score']==4) & (finalpppspatfamily['problem_patent']), 'ppp_score'] = 3
print(finalpppspatfamily.value_counts('ppp_score'))
# finalppps.loc[(finalppps['ppp_score'] == 4) & (finalppps['problem_magid'] == True), 'ppp_score'] = 3


In [ ]:
# Update finalppps with family-based rescoring
finalppps = finalppps.merge(finalpppspatfamily[['magid', 'patent_id', 'ppp_score']], on=['magid', 'patent_id'], how='left', suffixes=('', '_family'))
finalppps['ppp_score'] = finalppps['ppp_score_family'].fillna(finalppps['ppp_score'])
finalppps.drop(columns=['ppp_score_family'], inplace=True)

In [ ]:
# # Instead of setting ppp_score to 5, set a new flag "high_confidence"
# finalppps['high_confidence'] = False  # Initialize the new column
# finalppps.loc[(finalpppspatfamily['ppp_score'] >= 4) & 
#                         (finalppps['authors in common'] >= 2) & 
#                         (finalppps['fieldoverlap'] > 0.05) & 
#                         (finalppps['abstract_similarity'] > 0.6) & 
#                         # (finalppps['citeoverlapct'] > 0) & 
#                         # (finalppps['samegrant'] == 1) &
#                         (finalppps['daysdiffnoabs'] >= -180) & 
#                         (finalppps['proportion inventors'] > 25), 'high_confidence'] = True
# print(f"Number of high_confidence = True: {finalppps['high_confidence'].sum()}")
# # Set existing 5's to 4 before setting new ones to 5
# finalppps.loc[finalppps['ppp_score'] == 5, 'ppp_score'] = 4
# # Set ppp_score to 5 if high_confidence is True
# finalppps.loc[finalppps['high_confidence'] == True, 'ppp_score'] = 5
# print(finalppps.value_counts('ppp_score').sort_index())

In [ ]:
# handchecked = pd.read_csv(dataraw + 'PPP_Working Sheet - Responded.csv')
# handchecked = handchecked.rename(columns={'patentid': 'patent_id'})
# handchecked['magid'] = handchecked['magid'].astype(str)
# handchecked['patent_id'] = handchecked['patent_id'].astype(str)
# final_column = handchecked.columns[-1]
# if final_column not in handchecked.columns:
#     raise ValueError(f"Final column '{final_column}' not found in handchecked DataFrame.")
# categorized = handchecked[handchecked[final_column].isin(['A', 'B', 'C', 'D'])]
# handchecked = handchecked.merge(finalppps, on=['magid', 'patent_id'], how='inner')
# categorized = handchecked[handchecked[final_column].isin(['A', 'B', 'C', 'D'])]  # Re-filter after merge
# cross_tab = pd.crosstab(categorized[final_column], categorized['ppp_score']).sort_index(axis=1, ascending=False).T
# print("\nTransposed cross-tabulation of hand-checked scores (A/B/C/D) by ppp_score (4/3/2/1):")
# print(cross_tab)
# print("\nPercentages for each ppp_score:")
# for score in cross_tab.index:
#     row = cross_tab.loc[score]
#     total = row.sum()
#     if total > 0:
#         ab_count = row.get('A', 0) + row.get('B', 0)
#         cd_count = row.get('C', 0) + row.get('D', 0)
#         ab_percent = ab_count / total * 100
#         cd_percent = cd_count / total * 100
#         print(f"PPP Score {score}: Total = {total}, A/B = {ab_count} ({ab_percent:.1f}%), C/D = {cd_count} ({cd_percent:.1f}%)")
#         # Print magid and patent_id for C/D pairs at this score level
#         cd_pairs = categorized[(categorized[final_column].isin(['C', 'D'])) & (categorized['ppp_score'] == score)].drop_duplicates(subset=['magid', 'patent_id'])
#         if not cd_pairs.empty:
#             print(f"C/D pairs for PPP Score {score}:")
#             print(cd_pairs[['magid', 'patent_id']].to_string(index=False))
#         else:
#             print(f"No C/D pairs for PPP Score {score}")
#     else:
#         print(f"PPP Score {score}: No data")

In [ ]:
print(finalppps.value_counts('ppp_score').sort_index())

In [ ]:
writefile = True
if writefile is True:
    finalppps.drop(columns=['assessment', 'ppp_score_rf']).to_csv(datafin + 'finalppps.csv', index=False)


In [ ]:
# papertoembed = pd.read_csv(dataint + "papertoembed.tsv", sep='\t')
# papertoembed = papertoembed.rename({'id': 'magid', 'title': 'paper_title', 'abstract': 'paper_abstract'}, axis=1)
# papertoembed['magid'] = papertoembed['magid'].astype(str)
# papertoembed
# patenttoembed = pd.read_csv(dataint + "patenttoembed.tsv", sep='\t')
# patenttoembed = patenttoembed.rename({'id': 'patent_id', 'title': 'patent_title', 'abstract': 'patent_abstract'}, axis=1)

# patenttoembed
# df_pos_plagiarism_TIGHTER['byhand'].value_counts()
# df_pos_plagiarism_TIGHTER['plagiarism_percentage'].describe() 
# df_pos_plagiarism_TIGHTER
# df_pos_plagiarism2show = df_pos_plagiarism_TIGHTER[['magid', 'patent_id']]
# self_plagiarism_stats = pd.read_csv(dataint + 'finalppps_selfplagiarism.csv')
# self_plagiarism_stats['magid'] = self_plagiarism_stats['magid'].astype(str)
# df_pos_plagiarism2show = pd.merge(df_pos_plagiarism2show, self_plagiarism_stats, on=['magid', 'patent_id'], how='left')
# df_pos_plagiarism2show = pd.merge(df_pos_plagiarism2show, papertoembed, on='magid', how='left')
# df_pos_plagiarism2show = pd.merge(df_pos_plagiarism2show, patenttoembed, on='patent_id', how='left')
# df_pos_plagiarism2show = df_pos_plagiarism2show[['magid', 'patent_id', 'plagiarism_group_words_3', 'plagiarism_percentage', 'common_strings', 'paper_title', 'patent_title', 'paper_abstract', 'patent_abstract']]
# df_pos_plagiarism2show = df_pos_plagiarism2show.rename(columns={'plagiarism_percentage': 'selfplagpct', 'plagiarism_group_words_3': 'plag3'})
# df_pos_plagiarism2show = df_pos_plagiarism2show.dropna()
# pd.set_option('display.max_colwidth', None)
# df_pos_plagiarism2show.to_csv(dataint + "selfplagiarism_trainingsamples.tsv", index=False, sep='\t')
# df_pos_plagiarism2show

In [ ]:
dographs = True

In [ ]:
if dographs is True:
    import matplotlib.pyplot as plt
    import seaborn as sns

    finalppps2graph = finalppps.copy()
    # finalppps2graph = finalppps2graph[finalppps2graph['ppp_score'] >1]
    finalppps2graph = finalppps2graph[finalppps2graph['ppp_score'] == 4]

    # Create histogram
    plt.figure(figsize=(10,6))
    sns.histplot(data=finalppps2graph, x='daysdiffnoabs', bins=30,  edgecolor='black')
    plt.xlabel('Days Between Patent and Paper')
    plt.ylabel('Frequency')
    plt.xlim(-730, 730)
    plt.axvline(x=365, color='red', linestyle='--', label='365 days')
    plt.axvline(x=0, color='red', linestyle='--', label='same day')
    plt.title('Histogram of days between patent and paper')
    plt.grid(True, alpha=0.3)  # Add light grid
    plt.show()
    plt.savefig(graphoutputdir + '/daysdiffhiconf.png')
    # Basic statistics for context
    # print(finalppps['daysdiffnoabs'].describe())


In [ ]:
if dographs is True:
    import matplotlib.pyplot as plt
    import seaborn as sns

    finalppps2graph = finalppps.copy()
    finalppps2graph = finalppps2graph[finalppps2graph['ppp_score'] >1]
    # Create histogram
    plt.figure(figsize=(10,6))
    sns.histplot(data=finalppps2graph, x='daysdiffnoabs', bins=30,  edgecolor='black')
    plt.xlabel('Days Between Patent and Paper')
    plt.xlim(-730, 730)
    plt.ylabel('Frequency')
    plt.axvline(x=365, color='red', linestyle='--', label='365 days')
    plt.axvline(x=0, color='red', linestyle='--', label='same day')
    plt.title('Histogram of days between patent and paper')
    plt.grid(True, alpha=0.3)  # Add light grid
    plt.show()
    plt.savefig(graphoutputdir + '/daysdiffall.png')
    # Basic statistics for context
    # print(finalppps['daysdiffnoabs'].describe())


In [ ]:
if dographs is True:
    import matplotlib.pyplot as plt
    import seaborn as sns

    finalppps2graph = finalppps.copy()
    finalppps2graph = finalppps2graph[finalppps2graph['ppp_score'] >1]
    finalppps2graph = pd.merge(finalppps2graph, paperdates, on='magid', how='left')
    # Drop rows where paperpubdate is January 1st (regardless of year)
    finalppps2graph['paperpubdate'] = pd.to_datetime(finalppps2graph['paperpubdate'], errors='coerce')
    finalppps2graph = finalppps2graph[~((finalppps2graph['paperpubdate'].dt.month == 1) & (finalppps2graph['paperpubdate'].dt.day == 1))]
    print(finalppps2graph.shape)

    # Create histogram
    plt.figure(figsize=(10,6))
    sns.histplot(data=finalppps2graph, x='daysdiffnoabs', bins=30,  edgecolor='black')
    plt.xlabel('Days Between Patent and Paper')
    plt.xlim(-730, 730)
    plt.ylabel('Frequency')
    plt.axvline(x=365, color='red', linestyle='--', label='365 days')
    plt.axvline(x=0, color='red', linestyle='--', label='same day')
    plt.title('Histogram of days between patent and paper')
    plt.grid(True, alpha=0.3)  # Add light grid
    plt.show()
    plt.savefig(graphoutputdir + '/daysdiffall.png')
    # Basic statistics for context
    # print(finalppps['daysdiffnoabs'].describe())


## LLM adjustment (formerly applyLLMadjustment002)

A was already promoted to 5 above. Here: D -> 1, C -> 2; drop 1s; shift every score down by one (5/4/3/2 -> 4/3/2/1); optional readjust; keep the highest score per pair; write `finalpppsadjusted_prescored.csv`; score against the confidential author hand-checks; write the published `finalpppsadjusted.csv` (4 -> high, 3 -> mid; 2 = low and 1 are dropped).

In [ ]:
finalpppsllmadjusted = finalppps.copy()
no_match = finalpppsllmadjusted[(finalpppsllmadjusted['ppp_score'] == 4) & (finalpppsllmadjusted['assessment'].isna())]
print(f"Number of ppp_score=4 rows without a Grok assessment: {len(no_match)}")

finalpppsllmadjusted['ppp_score'] = np.select(
    [
        finalpppsllmadjusted['assessment'] == 'A',
        finalpppsllmadjusted['assessment'] == 'D',
        finalpppsllmadjusted['assessment'] == 'C'
    ],
    [5, 1, 2],
    default=finalpppsllmadjusted['ppp_score']  # keep original if not matching
)
finalpppsllmadjusted = finalpppsllmadjusted.drop(columns=['assessment', 'ppp_score_rf'])
finalpppsllmadjusted.value_counts('ppp_score').sort_values(ascending=False)


In [ ]:
finalpppsllmadjusted = finalpppsllmadjusted[finalpppsllmadjusted['ppp_score'] != 1].copy()
finalpppsllmadjusted['ppp_score'] = finalpppsllmadjusted['ppp_score'] - 1
finalpppsllmadjusted.value_counts('ppp_score').sort_values(ascending=False)


In [ ]:
if readjust is True:
    affected_rows = finalpppsllmadjusted[(finalpppsllmadjusted['ppp_score'] == 4) & (finalpppsllmadjusted['daysdiffnoabs'] < -180) & (finalpppsllmadjusted['authors in common'] < 3)].shape[0]
    print(f"Number of rows affected: {affected_rows}")
    finalpppsllmadjusted.loc[(finalpppsllmadjusted['ppp_score'] == 4) & (finalpppsllmadjusted['daysdiffnoabs'] < -180) & (finalpppsllmadjusted['authors in common'] < 3), 'ppp_score'] = 3


In [ ]:
finalpppsllmadjusted = (
    finalpppsllmadjusted[['magid', 'patent_id', 'ppp_score']]
    .sort_values('ppp_score', ascending=False)
    .drop_duplicates(['magid', 'patent_id'], keep='first')
    .reset_index(drop=True)
)
finalpppsllmadjusted.value_counts('ppp_score').sort_values(ascending=False)


In [ ]:
if writefile is True:
    finalpppsllmadjusted.to_csv(datafin + 'finalpppsadjusted_prescored.csv', index=False)


### Accuracy against the author hand-checks (confidential; skipped if `data/conf/authorhandcheck/` is absent)

In [ ]:
import os
from glob import glob

handcheck_dir = os.path.join(dataconf, 'authorhandcheck')
csv_paths = glob(os.path.join(handcheck_dir, '*.csv'))
if not csv_paths:
    print(f'WARNING: no author-validation files in {handcheck_dir} (confidential, not publicly distributed). Skipping accuracy scoring; the dataset itself is unaffected.')

handchecked_frames = []
for path in csv_paths:
    print(f"Reading file: {path}")
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=False)
    df = df.rename(columns={col: col.lower() for col in df.columns})

    if 'patentid' in df.columns:
        df = df.rename(columns={'patentid': 'patent_id'})
    if 'magid' not in df.columns or 'patent_id' not in df.columns:
        print(f"Skipping {path}: missing magid or patent_id")
        continue

    if 'response' not in df.columns:
        print(f"Skipping {path}: no 'response' column after normalization")
        continue

    df['magid'] = df['magid'].astype(str)
    df['patent_id'] = df['patent_id'].astype(str)
    handchecked_frames.append(df)

if not handchecked_frames:
    handchecked_frames = [pd.DataFrame(columns=['magid', 'patent_id', 'response'])]

handchecked = pd.concat(handchecked_frames, ignore_index=True)
handchecked = handchecked[['magid', 'patent_id', 'response']]
handchecked = handchecked[~(
    (handchecked['magid'] == '1986017883') &
    (handchecked['patent_id'] == '8088587') &
    (handchecked['response'].str.upper() == 'D')
)]
categorized = handchecked[handchecked['response'].isin(['A', 'B', 'C', 'D'])]
categorized = categorized.drop_duplicates()

duplicates = categorized[categorized.duplicated(['magid', 'patent_id'], keep=False)]
print(f"Duplicate magid/patent_id rows: {len(duplicates)}")
if len(duplicates):
    print(duplicates)

print(f"Categorized rows: {len(categorized)}")
categorized.value_counts('response').sort_values(ascending=False)


In [ ]:
handchecked = handchecked.merge(finalpppsllmadjusted, on=['magid', 'patent_id'], how='inner')
categorized = handchecked[handchecked['response'].isin(['A', 'B', 'C', 'D'])]  # Re-filter after merge
categorized = (
    categorized.sort_values('ppp_score', ascending=False).drop_duplicates(['magid', 'patent_id'], keep='first').reset_index(drop=True)
)

print(f"matched rows: {len(categorized)}")

cross_tab = pd.crosstab(categorized['response'], categorized['ppp_score']).sort_index(axis=1, ascending=False).T
print("\nTransposed cross-tabulation of hand-checked scores (A/B/C/D) by ppp_score (4/3/2/1):")
print(cross_tab)
print("\nPercentages for each ppp_score:")
for score in cross_tab.index:
    row = cross_tab.loc[score]
    total = row.sum()
    if total > 0:
        ab_count = row.get('A', 0) + row.get('B', 0)
        a_count = row.get('A', 0)
        cd_count = row.get('C', 0) + row.get('D', 0)
        bcd_count = row.get('B', 0) + row.get('C', 0) + row.get('D', 0)
        ab_percent = ab_count / total * 100
        a_percent = a_count / total * 100
        cd_percent = cd_count / total * 100
        bcd_percent = bcd_count / total * 100
        print(f"PPP Score {score}:   Total = {total}")
        print(f"  A/B = {ab_count} ({ab_percent:.2f}%), C/D = {cd_count} ({cd_percent:.2f}%)")
        print(f"  A = {a_count} ({a_percent:.2f}%), B/C/D = {bcd_count} ({bcd_percent:.2f}%)")
    else:
        print(f"PPP Score {score}: No data")


### Published dataset

In [ ]:
finalpppsllmadjusted2save = finalpppsllmadjusted.copy()
# drop if ppp_score is 1 or 2 (2 = 'low' confidence, excluded from the final output)
finalpppsllmadjusted2save = finalpppsllmadjusted2save[~finalpppsllmadjusted2save['ppp_score'].isin([1, 2])]
# convert ppp_score 4 to "high", 3 to "mid"
finalpppsllmadjusted2save['ppp_score'] = finalpppsllmadjusted2save['ppp_score'].map({4: 'high', 3: 'mid'})
finalpppsllmadjusted2save = finalpppsllmadjusted2save.rename(columns={'ppp_score': 'confidence_level'})
print(finalpppsllmadjusted2save.columns)
print(f"Total rows after adjustment: {len(finalpppsllmadjusted2save)}")
if writefile is True:
    finalpppsllmadjusted2save.to_csv(datafin + 'finalpppsadjusted.csv', index=False)
finalpppsllmadjusted2save['confidence_level'].value_counts()
